# Egypt Travel Guide Assistant — RAG pipeline notebook

**Graduation Project · Level 2 Summer Training · Core Track (text-based RAG)**

This notebook builds and evaluates the retrieval-augmented generation (RAG) pipeline behind the assistant:

`PDF guides → extract / OCR / clean → chunk → embed (Ollama) → Chroma vector store → retrieve → grounded, cited answer (Ollama LLM)`

**Domain:** Egypt tourism. **Corpus:** exactly four official guides (the only documents used):

| File | What it is |
|---|---|
| `Egypt_Tourism_Guide-Eng.pdf` | Egypt Tourism Guide brochure (61 pages) |
| `Alexandria-Eng.pdf` | *Alexandria & the White Med* brochure (100 pages) |
| `Egypt_Map-Eng.pdf` | *Egypt Map & Guide* poster (2 huge pages, with distance and temperature tables) |
| `Aswan_Museums_and_Sites_Guide-Eng.pdf` | *A Guide to Aswan's Museums and Archaeological Sites* (20 pages, **scanned — needs OCR**) |

**Before running:** Ollama must be running and the two models pulled (`ollama pull nomic-embed-text`, `ollama pull llama3.2`).
Use **Kernel → Restart & Run All**; the notebook is designed to run top to bottom.

In [3]:
import json, re, os, sys, time, math, shutil, statistics, collections, itertools, datetime as dt, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf                      # PDF text + page rendering (pip install pymupdf)
import chromadb
from chromadb.config import Settings
import ollama

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

# ---------------------------------------------------------------- paths
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR   = ROOT / "data" / "raw"
PROC_DIR  = ROOT / "data" / "processed"
STORE_DIR = ROOT / "data" / "vector_store"                 # notebook's working copy of the vector store
BACKEND_STORE = ROOT / "backend" / "data" / "vector_store"  # what the backend loads (written in 2.7)
DOCS_DIR  = ROOT / "docs"
for d in (PROC_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- models & parameters
OLLAMA_HOST  = os.getenv("OLLAMA_HOST", "http://localhost:11434")
EMBED_MODEL  = os.getenv("EMBED_MODEL", "nomic-embed-text")
DOC_PREFIX   = "search_document: "     # nomic-embed-text was trained with task prefixes
QUERY_PREFIX = "search_query: "
LLM_MODEL    = os.getenv("LLM_MODEL", "llama3.2")
COLLECTION   = "egypt_travel_guides"

CHUNK_SIZE       = 1200    # max characters per chunk when a page/section has to be split
CHUNK_OVERLAP    = 200     # characters repeated between consecutive chunks of a split page
PAGE_KEEP_LIMIT  = 1800    # pages/sections up to this size are kept whole (they are already topical units)
MIN_CHARS        = 60      # drop fragments shorter than this (captions, page furniture)
TOP_K            = 5

# registry of the four documents: file name -> metadata
DOCS = {
    "Egypt_Tourism_Guide-Eng.pdf":            dict(doc_id="egypt-tourism-guide", short="Egypt Tourism Guide",   title="Egypt Tourism Guide"),
    "Alexandria-Eng.pdf":                     dict(doc_id="alexandria-guide",    short="Alexandria Guide",      title="Alexandria & the White Med"),
    "Egypt_Map-Eng.pdf":                      dict(doc_id="egypt-map-guide",     short="Egypt Map & Guide",     title="Egypt Map & Guide"),
    "Aswan_Museums_and_Sites_Guide-Eng.pdf":  dict(doc_id="aswan-guide",         short="Aswan Guide",           title="A Guide to Aswan's Museums and Archaeological Sites"),
}
print("Project root:", ROOT)
print("Raw files found:", sorted(p.name for p in RAW_DIR.glob("*.pdf")))

Project root: C:\Users\user\Downloads\rag-assistant-app
Raw files found: ['Alexandria-Eng.pdf', 'Aswan_Museums_and_Sites_Guide-Eng.pdf', 'Egypt_Map-Eng.pdf', 'Egypt_Tourism_Guide-Eng.pdf']


# 2.1 Load & Inspect

First an **inventory** of every PDF: number of pages, size, how much text can be extracted, and — the key question from Phase 1 — *is the text really extractable, or is the file a scan that needs OCR?*

In [4]:
def inventory(pdf_path: Path) -> dict:
    doc = pymupdf.open(pdf_path)
    chars = [len(p.get_text().strip()) for p in doc]
    n = len(doc)
    return dict(
        file=pdf_path.name, format="PDF", pages=n,
        size_mb=round(pdf_path.stat().st_size / 1e6, 1),
        pages_with_text=sum(c >= 40 for c in chars),
        near_empty_pages=sum(c < 40 for c in chars),
        text_chars=sum(chars),
        chars_per_page=round(sum(chars) / n),
        embedded_fonts=any(p.get_fonts() for p in doc),
    )

inv = pd.DataFrame([inventory(RAW_DIR / f) for f in DOCS])
# a page-average of < 50 characters means "no usable text layer" -> scanned document
inv["needs_ocr"] = inv["chars_per_page"] < 50
inv

,file,format,pages,size_mb,pages_with_text,near_empty_pages,text_chars,chars_per_page,embedded_fonts,needs_ocr
0,Egypt_Tourism_Guide-Eng.pdf,PDF,61,23.1,46,15,37452,614,True,False
1,Alexandria-Eng.pdf,PDF,100,25.0,74,26,70716,707,True,False
2,Egypt_Map-Eng.pdf,PDF,2,10.5,2,0,62872,31436,True,False
3,Aswan_Museums_and_Sites_Guide-Eng.pdf,PDF,20,1.8,0,20,0,0,False,True


**What the inventory shows.** Three PDFs have a real text layer; the Aswan guide has *no embedded fonts and zero extractable characters* — it is a stack of page images, so it must go through OCR.
The near-empty pages in the other three files are cover pages, section dividers and full-page photographs (checked visually), so skipping them loses nothing.

### Extraction strategy per file
* **Text-layer PDFs** (Tourism guide, Alexandria, Map): extracted with PyMuPDF at *block* level, which keeps paragraphs together and preserves reading order better than raw page text. Cleaning: remove icon glyphs, page numbers, curved map labels (1-pt-wide "text" that would otherwise become single-letter noise), duplicated blocks, ligatures; join hyphenated line breaks; tag the address / phone / opening-hours lines that lost their icons.
* **Tables inside the Map poster** (temperatures, distances): the raw text is a stream of numbers that is meaningless to an embedding model. They are **parsed into natural-language facts** (one chunk per city).
* **Scanned Aswan guide:** OCR (RapidOCR, pure-pip) + layout reconstruction from box coordinates + dictionary-guided repair + a short list of manual fixes.

In [5]:
# ------------------------------------------------------------------ text-layer extraction
PHONE_RE = re.compile(r"^\+?\d{2,4}\s?[/\-]?\s?\d{3}\s?\d{3,4}$")                       # 03/480 9144
HOURS_RE = re.compile(r"^\d{1,2}([:.]\d{2})?\s?(am|pm)?\s?[-–]\s?\d{1,2}([:.]\d{2})?\s?(am|pm)$", re.I)   # 9am-4:30pm
COORD_RE = re.compile(r"^\d{1,3}°\s?\d{1,2}[’'′]\s?\d{1,2}(\.\d+)?[”\"″]?\s?[NSEW]$")     # 31°12'47.40" N


def clean_block(raw: str) -> str:
    """Clean one PyMuPDF text block and tag info lines (address/phone/hours)."""
    t = unicodedata.normalize("NFKC", raw)                       # 'ﬁ' -> 'fi', non-breaking spaces -> spaces
    t = re.sub(r"[\ue000-\uf8ff]", " ", t).replace("\t", " ")     # private-use icon glyphs
    t = re.sub(r"(?<=[a-z])-\n(?=[a-z])", "", t)                 # hyphenated line breaks
    lines = [re.sub(r"\s+", " ", l).strip() for l in t.split("\n")]
    lines = [l for l in lines if l]
    if not lines:
        return ""
    has_info = any(PHONE_RE.match(l) or HOURS_RE.match(l) for l in lines)
    out, para = [], []

    def flush():
        if para:
            out.append(" ".join(para)); para.clear()

    for i, l in enumerate(lines):
        if COORD_RE.match(l):
            continue                                              # GPS coordinates: noise for Q&A
        if PHONE_RE.match(l):
            flush(); out.append("Phone: " + l)
        elif HOURS_RE.match(l):
            flush(); out.append("Opening hours: " + l)
        elif has_info and i == 0:
            flush(); out.append("Address: " + l)
        elif i == 0 and len(lines) > 1 and l.isupper() and 3 <= len(l) <= 60:
            flush(); out.append(l)                                # heading kept on its own line
        elif l.startswith("•"):
            flush(); para.append(l)
        else:
            para.append(l)
    flush()
    return "\n".join(out)


def find_table_region(blocks):
    """Locate the temperature/distance tables of the Map poster (returns bbox + blocks in stream order)."""
    a = next((b for b in blocks if "Temperatures (Degrees" in b[4]), None)
    e = next((b for b in blocks if "Please note that this chart" in b[4]), None)
    if not (a and e):
        return None
    inside = [b for b in blocks if a[1] - 2 <= b[1] and b[3] <= e[3] + 2 and b[0] >= a[0] - 260]
    return dict(x0=min(b[0] for b in inside), y0=a[1] - 2, x1=max(b[2] for b in inside), y1=e[3] + 2, blocks=inside)


def extract_text_layer(pdf_path: Path, meta: dict):
    rows, table_blocks = [], []
    for pno, page in enumerate(pymupdf.open(pdf_path), 1):
        blocks = [b for b in page.get_text("blocks") if b[6] == 0]
        region = find_table_region(blocks)
        if region:
            table_blocks = region["blocks"]
        parts, seen = [], set()
        for x0, y0, x1, y1, txt, _, _ in blocks:
            if (x1 - x0) < 6 or (y1 - y0) < 3:                      # curved map labels (1-pt boxes)
                continue
            if region and x0 >= region["x0"] - 2 and y0 >= region["y0"] and y1 <= region["y1"] + 2:
                continue                                            # table cells: handled separately
            c = clean_block(txt)
            if len(re.sub(r"[^A-Za-z0-9]", "", c)) < 3 or re.fullmatch(r"\d{1,3}", c) or c in seen:
                continue
            seen.add(c)
            if (parts and "\n" not in parts[-1] and "\n" not in c and c[:1].islower()
                    and not re.search(r"[.!?:;”\"')]\s*$", parts[-1])):
                parts[-1] += " " + c                                # sentence continues in the next column/block
            else:
                parts.append(c)
        rows.append(dict(doc_id=meta["doc_id"], doc_short=meta["short"], page=pno, section=None,
                         kind="prose", method="text-layer", text="\n\n".join(parts)))
    return rows, table_blocks


page_rows, map_table_blocks = [], []
for fname, meta in DOCS.items():
    if fname.startswith("Aswan"):
        continue
    rows, tb = extract_text_layer(RAW_DIR / fname, meta)
    page_rows += rows
    if tb:
        map_table_blocks = tb
print(f"text-layer pages extracted: {len(page_rows)}")
print(f"table blocks captured from the Map poster: {len(map_table_blocks)}")

text-layer pages extracted: 163
table blocks captured from the Map poster: 16


### Tables → natural-language facts
The Map poster contains two tables: average monthly temperatures (7 cities) and a triangular distance matrix (18 places).
As raw text they come out as hundreds of loose numbers. Instead of feeding that to the embedding model, they are parsed into sentences such as
*"Cairo to Luxor: 721 km"* and *"Average monthly temperatures in Aswan … August: min 25, max 41"* — one chunk per city — so a question like *"How far is Siwa from Alexandria?"* can be answered from a single, self-contained passage.
The parser is **validated**: every distance row must contain exactly the expected number of values and the matrix must be symmetric.

In [6]:
MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
DIST_ROWS = ["Alexandria", "Aswan", "Bahariyya Oasis", "Cairo", "Dahab", "El Gouna", "El Quseir", "Fayoum Oasis", "Giza",
             "Hurghada", "Luxor", "Marsa Alam", "Nuweiba", "Sharm el-Sheikh", "Siwa Oasis", "Suez", "Taba"]
PLACES = ["Abu Simbel"] + DIST_ROWS          # column order of the matrix: Abu Simbel, Alexandria, ..., Suez (Taba has no column)


def parse_map_tables(table_blocks):
    text = "\n".join(b[4] for b in table_blocks)
    toks = [t.strip() for t in text.split("\n") if t.strip()]

    # ---- temperatures: <city> Mini <12 numbers> Max <12 numbers>
    temps, i = {}, toks.index("Dec") + 1
    while i < len(toks) and not toks[i].startswith("Distances"):
        city = toks[i]; i += 1
        assert toks[i] == "Mini" and toks[i + 13] == "Max", f"unexpected temperature layout near {city}"
        mins = [int(x) for x in toks[i + 1:i + 13]]
        maxs = [int(x) for x in toks[i + 14:i + 26]]
        temps[city] = (mins, maxs); i += 26

    # ---- distances: triangular matrix, row k has k+1 values (columns: Abu Simbel, Alexandria, ...)
    j = next(k for k, t in enumerate(toks) if t.startswith("Distances"))
    rest = " ".join(toks[j + 1:])
    rest = rest[rest.index("Alexandria 1466"):]
    dist, pos = {}, 0
    for k, name in enumerate(DIST_ROWS):
        pat = re.compile(re.escape(name) + r"\s+" + r"\s+".join([r"(\d+)"] * (k + 1)))
        m = pat.match(rest, pos)
        assert m, f"distance row not parsed: {name}"
        for col, val in zip(PLACES[:k + 1], m.groups()):
            dist[(name, col)] = dist[(col, name)] = int(val)
        pos = m.end()
        while pos < len(rest) and rest[pos] == " ":
            pos += 1
    return temps, dist


temps, dist = parse_map_tables(map_table_blocks)
# ---- validation
assert len(temps) == 7 and all(len(v[0]) == 12 and len(v[1]) == 12 for v in temps.values())
assert all(mn <= mx for mins, maxs in temps.values() for mn, mx in zip(mins, maxs)), "min > max somewhere"
assert len(dist) == 2 * (18 * 17 // 2), f"expected 153 pairs, got {len(dist) // 2}"
assert dist[("Giza", "Cairo")] == 8 and dist[("Cairo", "Alexandria")] == 224      # sanity checks against known values
print(f"parsed {len(temps)} temperature rows and {len(dist)//2} distance pairs — validation passed")


def table_fact_rows():
    rows, short = [], DOCS["Egypt_Map-Eng.pdf"]["short"]
    for city, (mins, maxs) in temps.items():
        cells = "; ".join(f"{m}: min {a}, max {b}" for m, a, b in zip(MONTHS, mins, maxs))
        hot = MONTHS[int(np.argmax(maxs))]
        text = (f"Average monthly temperatures in {city} in degrees Celsius (Egypt Map & Guide temperature table). {cells}. "
                f"Hottest month: {hot} (max {max(maxs)} °C). Coolest month: {MONTHS[int(np.argmin(mins))]} (min {min(mins)} °C).")
        rows.append(dict(doc_id="egypt-map-guide", doc_short=short, page=2, section=f"Temperatures in {city}",
                         kind="table", method="table-parse", text=text))
    for city in PLACES:
        others = [p for p in PLACES if p != city]
        cells = "; ".join(f"{city} to {p}: {dist[(city, p)]} km" for p in others)
        text = f"Distance between {city} and other Egyptian destinations, in kilometres (Egypt Map & Guide distance table). {cells}."
        rows.append(dict(doc_id="egypt-map-guide", doc_short=short, page=2, section=f"Distances from {city}",
                         kind="table", method="table-parse", text=text))
    return rows

fact_rows = table_fact_rows()
print(fact_rows[0]["text"][:330], "...\n")
print(fact_rows[7]["text"][:330], "...")

parsed 7 temperature rows and 153 distance pairs — validation passed
Average monthly temperatures in Cairo in degrees Celsius (Egypt Map & Guide temperature table). January: min 11, max 19; February: min 12, max 24; March: min 13, max 25; April: min 14, max 26; May: min 15, max 28; June: min 17, max 32; July: min 20, max 35; August: min 22, max 37; September: min 20, max 30; October: min 18, max  ...

Distance between Abu Simbel and other Egyptian destinations, in kilometres (Egypt Map & Guide distance table). Abu Simbel to Alexandria: 1466 km; Abu Simbel to Aswan: 260 km; Abu Simbel to Bahariyya Oasis: 1276 km; Abu Simbel to Cairo: 1242 km; Abu Simbel to Dahab: 1544 km; Abu Simbel to El Gouna: 770 km; Abu Simbel to El Quseir ...


### OCR of the scanned Aswan guide
Pipeline for every page: render at 300 dpi → RapidOCR → **rebuild layout from box coordinates** → repair text → apply manual fixes.

* **Layout from coordinates.** The OCR returns loose boxes. They are grouped into lines by vertical position; the title is the tallest line at the top; the *Address / Opening hours / Ticket prices* block at the bottom is parsed separately. The ticket-price table is reconstructed geometrically — each price is assigned to the *Visitor* or *Student* header above it — and written as `Ticket prices: Visitor 450 EGP, Student 230 EGP.` (a plain left-to-right read scrambles which number belongs to whom).
* **Missed lines.** The detector sometimes silently skips a whole line. When the vertical gap between two body lines is larger than the normal line pitch, that band is re-OCR'd at higher zoom (it recovered a whole sentence on page 11 of this document).
* **Noise.** Decorative triangles on the museum façade and logo art are filtered (garbage-box heuristic), as are tiny map labels.
* **Repair.** OCR merges words (`Donotmiss`) and confuses letters (`c`↔`e`, `u`↔`n`, `Nckhbct`). A lexicon built from the three text-layer documents plus an English word list is used to (1) fix words by substituting confusable letters and (2) split glued words. Ambiguity is resolved in favour of words that occur elsewhere in the same document.
* **Manual fixes.** A small, explicit rule list handles what a dictionary cannot (found by reading every OCR'd page next to its image).

In [7]:
import itertools, difflib
import wordninja                      # pip install wordninja (word-frequency list for the repair step)
from PIL import Image

OCR_DPI = 300
_engine = None


def get_ocr():
    global _engine
    if _engine is None:
        from rapidocr_onnxruntime import RapidOCR      # imported lazily: only needed when OCR really runs
        _engine = RapidOCR()
    return _engine


def _run(img):
    result, _ = get_ocr()(img)
    out = []
    for pts, text, conf in (result or []):
        xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
        out.append(dict(x0=min(xs), y0=min(ys), x1=max(xs), y1=max(ys), text=text.strip(), conf=float(conf)))
    return out


def group_lines(boxes):
    boxes = sorted(boxes, key=lambda b: (b["y0"] + b["y1"]) / 2)
    lines = []
    for b in boxes:
        yc = (b["y0"] + b["y1"]) / 2
        h = b["y1"] - b["y0"]
        for ln in lines:
            if abs(yc - ln["yc"]) < 0.5 * max(h, ln["h"]):
                ln["boxes"].append(b)
                ln["yc"] = statistics.mean((bb["y0"] + bb["y1"]) / 2 for bb in ln["boxes"])
                ln["h"] = max(ln["h"], h)
                break
        else:
            lines.append(dict(yc=yc, h=h, boxes=[b]))
    for ln in lines:
        ln["boxes"].sort(key=lambda b: b["x0"])
        ln["text"] = " ".join(b["text"] for b in ln["boxes"])
        ln["x0"] = min(b["x0"] for b in ln["boxes"])
        ln["y0"] = min(b["y0"] for b in ln["boxes"])
    lines.sort(key=lambda l: l["yc"])
    return lines


def _v_overlap(a, b):
    inter = min(a["y1"], b["y1"]) - max(a["y0"], b["y0"])
    if inter <= 0:
        return 0.0
    xin = min(a["x1"], b["x1"]) - max(a["x0"], b["x0"])
    if xin <= 0:
        return 0.0
    return inter / min(a["y1"] - a["y0"], b["y1"] - b["y0"])


def ocr_page(page, dpi=OCR_DPI):
    """OCR one page; re-OCR any band where a text line was probably missed."""
    pix = page.get_pixmap(dpi=dpi)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)[:, :, :3]
    boxes = _run(img)
    body = [b for b in boxes if len(b["text"]) > 40]
    added = 0
    if len(body) >= 4:
        lines = group_lines(body)
        pitches = [b["y0"] - a["y0"] for a, b in zip(lines, lines[1:])]
        pitch = statistics.median(pitches)
        for a, b in zip(lines, lines[1:]):
            gap = b["y0"] - a["y0"]
            if gap > 1.6 * pitch:                                   # a whole line is missing here
                y0 = int(a["y0"] + pitch * 0.8); y1 = int(b["y0"] + pitch * 0.15)
                crop = Image.fromarray(img[max(y0 - 15, 0):y1 + 15, :, :])
                pad = Image.new("RGB", (crop.width, crop.height + 80), "white"); pad.paste(crop, (0, 40))
                off = max(y0 - 15, 0) - 40
                for nb in _run(np.array(pad)):
                    if nb["conf"] < 0.85 or len(nb["text"]) <= 15:
                        continue
                    nb["y0"] += off; nb["y1"] += off
                    if any(_v_overlap(nb, ob) > 0.25 for ob in boxes):    # already detected: skip duplicate
                        continue
                    boxes.append(nb); added += 1
    return dict(boxes=boxes, size=(pix.width, pix.height), gap_fills=added)


# ------------------------------------------------------------------ vocab
SHORT_OK = {"a", "i", "of", "to", "in", "on", "at", "or", "is", "it", "as", "by", "be", "do",
            "if", "so", "no", "an", "we", "he", "me", "my", "up", "us", "go"}
_wc = wordninja.DEFAULT_LANGUAGE_MODEL._wordcost
CONFUSABLE = {"c": "e", "e": "c", "u": "n", "n": "u", "i": "l", "l": "i", "0": "o"}


class Lexicon:
    def __init__(self, domain_words, target_words=()):
        self.domain = {w.lower() for w in domain_words}
        self.targets = {w.lower() for w in target_words}

    def known(self, w):
        w = w.lower()
        if w in SHORT_OK or w in self.domain:
            return True
        if w not in _wc:
            return False
        return len(w) >= 6 or (len(w) >= 3 and _wc[w] < 12.5)

    def cost(self, w):
        w = w.lower()
        if w in _wc:
            return _wc[w]
        return 9.0 if w in self.domain else None

    def segment(self, s):
        """Viterbi word segmentation using only known words. Returns list or None."""
        s = s.lower(); n = len(s)
        best = [0.0] + [math.inf] * n
        back = [0] * (n + 1)
        repl = {}
        for i in range(1, n + 1):
            for j in range(max(0, i - 16), i):
                w = s[j:i]
                if len(w) < 2:
                    continue
                if self.known(w):
                    c = self.cost(w) or 9.0
                    r = None
                elif len(w) >= 4:
                    r = self.confusable_fix(w)
                    if not r:
                        continue
                    c = (self.cost(r) or 9.0) + 2.5
                else:
                    continue
                c += best[j] + 0.5
                if c < best[i]:
                    best[i], back[i] = c, j
                    repl[(j, i)] = r or w
        if best[n] == math.inf:
            return None
        out, i = [], n
        while i > 0:
            out.append(repl[(back[i], i)]); i = back[i]
        return list(reversed(out))

    def confusable_fix(self, w):
        low = w.lower()
        idx = [i for i, ch in enumerate(low) if ch in CONFUSABLE]
        if not idx or len(idx) > 10:
            return None
        cands = set()
        for k in (1, 2):
            for combo in itertools.combinations(idx, k):
                chars = list(low)
                for i in combo:
                    chars[i] = CONFUSABLE[chars[i]]
                cand = "".join(chars)
                if self.known(cand) or cand in self.targets:
                    cands.add(cand)
            if cands:
                break                                    # prefer the fewest substitutions
        if len(cands) == 1:
            return cands.pop()
        if len(cands) > 1:                               # ambiguous: pick the one seen in the document itself
            t = [c for c in cands if c in self.targets]
            if len(t) == 1:
                return t[0]
        return None

    def repair_word(self, w):
        if len(w) < 3 or self.known(w):
            return w
        if re.search(r"[a-z][A-Z]", w):                  # camelCase glue: 'thePredynastic'
            parts = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", w).split(" ")
            return " ".join(self.repair_word(p) for p in parts)
        fix = self.confusable_fix(w)
        if fix:
            return _match_case(w, fix)
        seg = self.segment(w)
        if seg and len(seg) > 1 and all(len(p) >= 3 or p in SHORT_OK for p in seg):
            return _match_case(w, " ".join(seg))
        return w


def _match_case(orig, new):
    if orig.isupper() and len(orig) > 1:
        return new.upper()
    if orig[0].isupper():
        return new[0].upper() + new[1:]
    return new


ROMAN_AFTER = r"(Rameses|Ramses|Ptolemy|Amenhotep|Thutmose|Sety|Seti|Nectanebo)"


def regex_fixes(s):
    s = s.replace("，", ",")
    s = re.sub(r"\b0(?=[A-Za-z]{3,})", "O", s)                                  # 0peningHours
    s = re.sub(r"(?<=[a-z])'s(?=[a-z]{3,})", "'s ", s)                          # Egypt'smost
    s = re.sub(r"(?<=[a-z])\.(?=[A-Z])", ". ", s)                                # site.This
    s = re.sub(r"(?<=[a-z\)\d]),(?=[A-Za-z])", ", ", s)                          # Nile,and  1997,has
    s = re.sub(r"(?<=\d)\.(?=[A-Za-z]{2,})", ". ", s)                             # 1860.and
    s = re.sub(r"(?<=[a-z])(?=(1[0-9]{3}|20[0-9]{2})\b)", " ", s)                  # in1960 -> in 1960
    s = re.sub(r"(\d)(?=metres\b)", r"\1 ", s)
    s = re.sub(r"(?<=[A-Z])'S(?=[A-Z][a-z])", "'S ", s)                          # UNESCO'SWorld
    s = re.sub(r"(?<=[a-z])'(?=[a-z]{3,}\b)", "' ", s) if False else s
    s = re.sub(r"(?<=[a-z]);(?=[A-Za-z])", "; ", s)
    s = re.sub(r"(\d)o(?=\s*km\b)", r"\g<1>0", s)                                # 7o km
    s = re.sub(r"(\d)\s*km\s*(?=[a-z])", r"\1 km ", s)
    s = re.sub(r"(\d)\s*km\b", r"\1 km", s)
    s = re.sub(r"(\d)(BC|AD)\b", r"\1 \2", s)
    s = re.sub(r"\)(?=[A-Za-z])", ") ", s)
    s = re.sub(r"(?<=[A-Za-z])\((?=[c\d])", " (", s)
    s = re.sub(r"\b(\d+)ed\b", r"\1nd", s)                                       # 22ed -> 22nd
    s = re.sub(r"\b1l(th)\b", r"11\1", s)
    s = re.sub(rf"\b{ROMAN_AFTER}\s+([IlHV1]{{1,3}})(?=[a-z]|\b)",
               lambda m: f"{m.group(1)} " + re.sub(r"[IlH1]", "I", m.group(2)), s)
    s = re.sub(r"(\d{1,2}[:.]\d{2})\s*([ap]m)", r"\1 \2", s, flags=re.I)
    s = re.sub(r"\s*[-–—]\s*(?=\d{1,2}[:.]\d{2}\s*[ap]m)", " – ", s, flags=re.I)
    return s


def repair_text(text, lex):
    def fix_tok(m):
        return lex.repair_word(m.group(0))
    text = regex_fixes(text)
    text = re.sub(r"[A-Za-z]{3,}", fix_tok, text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


# ------------------------------------------------------- page structure
def norm_key(s):
    return re.sub(r"[^a-z]", "", s.lower())


def fuzzy_starts(s, label, thr=0.72):
    k = norm_key(s)[: len(label)]
    return difflib.SequenceMatcher(None, k, label).ratio() >= thr


PRICE_RE = re.compile(r"^(\d{2,4})E?G?P$", re.I)
LOGO_RE = re.compile(r"LAND\s*OF\s*GOLD|^AS[NW]AN$|^ASN|oijlueji|^unesco", re.I)


def is_garbage(t):
    letters = re.sub(r"[^A-Za-z]", "", t)
    if len(letters) >= 8 and letters.isupper():
        top = max(letters.count(c) for c in set(letters))
        return top / len(letters) >= 0.4
    return False


def parse_prices(boxes):
    heads = [b for b in boxes if norm_key(b["text"]) in ("visitor", "student")]
    prices = [b for b in boxes if PRICE_RE.match(b["text"].replace(" ", ""))]
    used = {id(b) for b in heads} | {id(b) for b in prices}
    if not heads or not prices:
        return [], set()
    heads.sort(key=lambda b: (b["x0"] + b["x1"]) / 2)
    assign = {}
    for p in prices:
        pc = (p["x0"] + p["x1"]) / 2
        above = [h for h in heads if h["y1"] <= p["y0"] + 10]
        if not above:
            continue
        h = min(above, key=lambda h: (abs((h["x0"] + h["x1"]) / 2 - pc), p["y0"] - h["y1"]))
        assign[id(h)] = re.sub(r"[^\d]", "", p["text"]) + " EGP"
    tables, cur = [], None
    for h in heads:
        kind = "Visitor" if norm_key(h["text"]) == "visitor" else "Student"
        if kind == "Visitor" or cur is None:
            cur = dict(label=None, vals={}, head_x=h["x0"], head_y=h["y0"])
            tables.append(cur)
        if id(h) in assign:
            cur["vals"][kind] = assign[id(h)]
    for i, t in enumerate(tables[1:], start=1):
        prev_x = tables[i - 1]["head_x"]
        for b in boxes:
            if id(b) in used:
                continue
            if abs(b["y0"] - t["head_y"]) < 40 and prev_x + 100 < b["x0"] < t["head_x"]:
                t["label"] = b["text"]; used.add(id(b))
    return [(t["label"], t["vals"]) for t in tables if t["vals"]], used


def join_paragraph(lines):
    out = ""
    for t in lines:
        if out.endswith("-") and t[:1].islower():
            out = out[:-1] + t
        elif out:
            out += " " + t
        else:
            out = t
    return out


def build_page_text(page_ocr, lex, min_line_chars=0):
    boxes, (W, H) = page_ocr["boxes"], page_ocr["size"]
    body_like = [b for b in boxes if len(b["text"]) > 40]
    med_h = statistics.median((b["y1"] - b["y0"]) for b in body_like) if body_like else None
    kept = [b for b in boxes if b["conf"] >= 0.4 and b["text"] and not is_garbage(b["text"]) and not LOGO_RE.search(b["text"])
            and (med_h is None or (b["y1"] - b["y0"]) >= 0.55 * med_h)]
    kept = [b for b in kept if not (re.fullmatch(r"\d{1,2}", b["text"]) and b["y0"] > 0.9 * H)]
    tables, used = parse_prices(kept)
    rest = [b for b in kept if id(b) not in used and not fuzzy_starts(b["text"], "ticketprices", 0.8)]
    lines = group_lines(rest)
    if min_line_chars:
        lines = [l for l in lines if len(l["text"]) >= min_line_chars]

    idx_addr = next((i for i, l in enumerate(lines) if fuzzy_starts(l["text"], "address", 0.8)), None)
    idx_open = next((i for i, l in enumerate(lines) if fuzzy_starts(l["text"], "openinghours", 0.72)), None)
    start = idx_addr
    if start is None and idx_open is not None:                 # 'Address:' label was not detected
        start = idx_open
        while start > 0 and lines[start - 1]["x0"] > 0.15 * W:
            start -= 1
    head_lines = lines[:start] if start is not None else lines
    info_lines = lines[start:] if start is not None else []

    title = None
    if start is not None and head_lines:
        big = max(head_lines[:3], key=lambda l: l["h"])
        if med_h and big["h"] > 1.2 * med_h and len(big["text"]) < 60:
            title = big["text"]; head_lines = [l for l in head_lines if l is not big]
    body = join_paragraph([l["text"] for l in head_lines])

    info = []
    for i, l in enumerate(info_lines):
        t = l["text"]
        if fuzzy_starts(t, "address", 0.8):
            t = "Address: " + re.sub(r"^\s*\S*\s*[:;]?\s*", "", t, count=1) if ":" not in t[:12] else "Address: " + t.split(":", 1)[1].strip()
        elif fuzzy_starts(t, "openinghours", 0.72):
            t = "Opening hours: " + (t.split(":", 1)[1].strip() if ":" in t[:20] else t[12:].strip())
        elif start is not None and i == 0:
            t = "Address: " + t
        info.append(t)
    # continuation of a wrapped address line
    merged = []
    for t in info:
        if merged and not re.match(r"(Address|Opening hours|On )", t) and merged[-1].startswith("Address"):
            merged[-1] += " " + t
        else:
            merged.append(t)
    info = merged
    if tables:
        parts = []
        for label, vals in tables:
            s = ", ".join(f"{k} {v}" for k, v in vals.items())
            parts.append(f"{label}: {s}" if label else s)
        info.append("Ticket prices: " + ". ".join(parts) + ".")
    chunks = []
    if title:
        chunks.append(title)
    if body:
        chunks.append(body)
    if info:
        chunks.append("\n".join(info))
    raw = "\n\n".join(chunks)
    return repair_text(raw, lex) if raw else ""

In [8]:
# Residual OCR errors found by reading every OCR'd page next to its page image.
# Each rule is (regex, replacement). Generic repairs (glued words, c/e and u/n confusions, prices, hours)
# are done by code; these are the few cases a dictionary cannot resolve.
MANUAL_FIXES = [
    (r"Unfnished", "Unfinished"),
    (r"\s*Archaeological and Touristic Attractions$", ""),                 # map caption on page 2
    (r"from Arabiekummound.and.Ombo.,\s*which", "from Arabic kum 'mound' and 'Ombo', which"),
    (r"\(City\)'\.The", "(City)'. The"),
    (r"tombs arc those", "tombs are those"),
    (r"do med tomb", "domed tomb"),
    (r"al-Hawa\(qubba meaning dome in Arabic\)", "al-Hawa (qubba, meaning 'dome' in Arabic)"),
    (r"bankofthc Nile", "bank of the Nile"),
    (r"the Gh or 7h century", "the 6th or 7th century"),
    (r"pilgrims'rooms", "pilgrims' rooms"),
    (r"Stcla", "Stela"),
    (r"1960by UNESCO", "1960 by UNESCO"),
    (r"nasser-south Aswan", "Nasser - south Aswan"),
    (r"lake nasser-behind", "Lake Nasser - behind"),
    (r"Isis and Scrap is", "Isis and Serapis."),
    (r"Thut mose Ill", "Thutmose III"),
    (r"temple ofal-Derr", "temple of al-Derr"),
    (r"Pen nut", "Pennut"),
    (r"BC\)\.Because", "BC). Because"),
    (r"Horus Sema- ?Tawy'Hlorus-Who-Uni ?tes-The-Two-Lands'\)\.?", "Horus Sema-Tawy 'Horus-Who-Unites-The-Two-Lands')."),
    (r"Lake nasser", "Lake Nasser"),
    (r"Banks lake", "Banks Lake"),
    (r"UNEsc ?O", "UNESCO"),
    (r"apleasantvisit", "a pleasant visit"),
    (r"opcninghours on:mota\.gov\.eg mota\.gov\.eg egymonuments\.com experienceegypt\.eg",
     "opening hours on: mota.gov.eg. Websites: mota.gov.eg, egymonuments.com, experienceegypt.eg"),
    (r"\bAvisit\b", "A visit"),
    (r"IRameses", "I, Rameses"),
    (r"\bmother hood\b", "motherhood"),
    (r"Lands'\)\.One", "Lands'). One"),
    (r"workers'inscriptions", "workers' inscriptions"),
    (r"Rameses Lime ren ptah, and Rameses Iii and", "Rameses II, Merenptah, and Rameses III, and"),
    (r"al-Mas alla", "al-Masalla"),
]


OCR_PAGE_RULES = {1: "skip",   # cover: logo art only, OCR gives garbage
                  2: 35}       # intro page: keep only lines >= 35 chars (drops small map labels)

ASWAN_FILE = "Aswan_Museums_and_Sites_Guide-Eng.pdf"
ASWAN_META = DOCS[ASWAN_FILE]
OCR_CACHE  = PROC_DIR / f"{ASWAN_META['doc_id']}_ocr_boxes.json"
FORCE_OCR  = False        # set True to ignore the cache and re-run OCR (~2 minutes)


def ocr_document(pdf_path: Path, force=False):
    if OCR_CACHE.exists() and not force:
        print(f"using cached OCR boxes: {OCR_CACHE.name}")
        return json.load(open(OCR_CACHE, encoding="utf-8"))
    print("running OCR (about 2 minutes)...")
    res = []
    for k, page in enumerate(pymupdf.open(pdf_path), 1):
        res.append(ocr_page(page)); print(f"  page {k}/20", end="\r")
    json.dump(res, open(OCR_CACHE, "w", encoding="utf-8"))
    return res


ocr_raw = ocr_document(RAW_DIR / ASWAN_FILE, FORCE_OCR)

# vocabulary from the text-layer documents = domain lexicon (place names, Egyptological terms...)
domain_words = set()
for r in page_rows:
    domain_words.update(w.lower() for w in re.findall(r"[A-Za-z]{2,}", r["text"]))
ocr_freq = collections.Counter()
for p in ocr_raw:
    for b in p["boxes"]:
        ocr_freq.update(w.lower() for w in re.findall(r"[A-Za-z]{4,}", b["text"]))
lexicon = Lexicon(domain_words, {w for w, c in ocr_freq.items() if c >= 2})

aswan_rows, skipped = [], []
for pno, p in enumerate(ocr_raw, 1):
    rule = OCR_PAGE_RULES.get(pno)
    if rule == "skip":
        skipped.append(pno); continue
    text = build_page_text(p, lexicon, min_line_chars=rule if isinstance(rule, int) else 0)
    for pat, rep in MANUAL_FIXES:
        text = re.sub(pat, rep, text)
    first = text.split("\n\n")[0] if "Address:" in text else None          # site name = title line of a site page
    aswan_rows.append(dict(doc_id=ASWAN_META["doc_id"], doc_short=ASWAN_META["short"], page=pno,
                           section=first if first and len(first) < 60 else None,
                           kind="prose", method="ocr", text=text))
print(f"OCR'd pages kept: {len(aswan_rows)} (skipped: {skipped}); gap-fill re-OCR recovered {sum(p['gap_fills'] for p in ocr_raw)} missed lines")
print("\n--- sample: page 4 ---\n" + aswan_rows[2]["text"])

using cached OCR boxes: aswan-guide_ocr_boxes.json
OCR'd pages kept: 19 (skipped: [1]); gap-fill re-OCR recovered 1 missed lines

--- sample: page 4 ---
Temple of Edfu

By visiting the temple of Edfu, which lies on the west bank of the Nile, you can explore one of the most completely preserved ancient Egyptian temples. Built during the Ptolemaic Period (305-30 BC), it is dedicated to the gods Horus, his wife Hathor, and their son Harsomtus (Horus Sema-Tawy 'Horus-Who-Unites-The-Two-Lands'). One of the reasons for the exceptional preservation of this temple is the fact that most of it was buried under sand, until Auguste Mariette cleared it in 1860. and restoration of some parts Fascinating scenes adorn the walls, including those in the inner and outer hypostyle halls that show the temple foundation rituals being performed by the king. And make sure you don't miss the scenes depicting the Beautiful Feast of Behdet, for which the cult statue of the goddess hathor sailed along distance fr

In [9]:
# ------------------------------------------------------------------ assemble the page table
pages = pd.DataFrame(page_rows + aswan_rows + fact_rows)
pages["section"] = pages["section"].fillna("")
pages["n_chars"] = pages["text"].str.len()
pages["n_words"] = pages["text"].str.split().str.len()
pages.to_json(PROC_DIR / "pages.jsonl", orient="records", lines=True, force_ascii=False)

summary = (pages.assign(usable=pages["n_chars"] >= MIN_CHARS)
                .groupby(["doc_short", "method"])
                .agg(pages=("page", "count"), usable=("usable", "sum"), total_chars=("n_chars", "sum"),
                     median_chars=("n_chars", "median")))
display(summary)
print("total documents:", pages["doc_id"].nunique(), "| source pages in PDFs:", int(inv["pages"].sum()),
      "| extracted units:", len(pages), "| total characters:", f"{pages['n_chars'].sum():,}")

pages  usable  total_chars  median_chars
doc_short           method                                               
Alexandria Guide    text-layer     100      70        66726         717.0
Aswan Guide         ocr             19      19        15956         863.0
Egypt Map & Guide   table-parse     25      25        14449         573.0
                    text-layer       2       2        60382       30191.0
Egypt Tourism Guide text-layer      61      45        36761         514.0

total documents: 4 | source pages in PDFs: 183 | extracted units: 207 | total characters: 194,274


### 2.1 — Answers

* **How many documents / pages?** 4 documents, **183 pages** in total (Tourism Guide 61, Alexandria 100, Map poster 2 very large sheets, Aswan guide 20). After extraction there are **207 text units** (163 text-layer pages, 19 OCR'd pages, 25 table-fact chunks) with about **194,000 characters**.
* **Which formats?** All four are PDF (three with a text layer, one scanned).
* **Which files needed OCR?** Only the **Aswan guide** — 20 scanned pages, no fonts, zero extractable characters. 19 pages were OCR'd; the cover (logo artwork only) was skipped.
* **Which files failed to parse?** None. No corrupted or encrypted files.
* **Pages with no usable text:** 41 in the text-layer PDFs (15 in the Tourism Guide, 26 in Alexandria) — covers, section dividers, full-page photographs and an image-only Alexandria street map. Their content is not searchable (known limitation: the legend of that map is an image).
* **Messy data that had to be cleaned:** multi-column poster layout with paragraphs that continue across blocks; temperature/distance tables that extract as loose numbers; curved map labels that extract as single letters; icon glyphs in front of address/phone/hours lines; hyphenated line breaks and ligatures; page numbers; and — for the scan — glued words, confused letters, a scrambled price table, a skipped text line and decorative artwork read as text (all handled above).

# 2.2 Chunking Strategy

**Strategy: section-aware, size-bounded chunks.** These guides are *brochures*: every page (or poster article) is already one topical unit — "Temple of Edfu", "Hurghada", "Emergency numbers". Cutting through a unit is what breaks RAG in practice (for the Aswan guide it would separate a site's name from its ticket prices).

1. A page/section of **≤ 1,800 characters is kept whole** (one chunk).
2. Longer pages are split at paragraph, then sentence boundaries into chunks of **≤ 1,200 characters with 200 characters of overlap** (whole sentences are carried over).
3. Table facts (one city each) are already atomic chunks.
4. Fragments under 60 characters (photo captions, page furniture) are dropped.
5. Every chunk is *embedded together with a header* — document, section title (e.g. "Temple of Edfu") and page — so the vector always knows which place/site it is about, even for the 2nd half of a split page.

In [10]:
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9“\"‘'(•])")


def to_units(text, size):
    """Paragraphs -> list of (text, paragraph_id); paragraphs longer than `size` are cut into sentences."""
    units = []
    for pid, para in enumerate(p for p in text.split("\n\n") if p.strip()):
        if len(para) <= size:
            units.append((para, pid)); continue
        for s in SENT_SPLIT.split(para):
            while len(s) > size:                                   # pathological sentence: cut at a space
                cut = s.rfind(" ", 0, size); cut = cut if cut > 0 else size
                units.append((s[:cut], pid)); s = s[cut:].lstrip()
            if s:
                units.append((s, pid))
    return units


def join_units(units):
    out = ""
    for k, (t, pid) in enumerate(units):
        out += (("\n\n" if pid != units[k - 1][1] else " ") if k else "") + t
    return out


def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP, keep_limit=PAGE_KEEP_LIMIT):
    text = text.strip()
    if len(text) <= keep_limit:
        return [text]
    units, chunks, cur = to_units(text, size), [], []
    for u in units:
        if cur and len(join_units(cur + [u])) > size:
            chunks.append(join_units(cur))
            carry, total = [], 0                                    # carry trailing sentences as overlap
            for x in reversed(cur):
                if total + len(x[0]) > overlap:
                    break
                carry.insert(0, x); total += len(x[0])
            cur = carry
        cur.append(u)
    if cur:
        chunks.append(join_units(cur))
    return chunks


def source_label(short, section, page):
    return f"{short} – {section} (p. {page})" if section else f"{short} (p. {page})"


chunks = []
for r in pages.to_dict("records"):
    pieces = [r["text"]] if r["kind"] == "table" else chunk_text(r["text"])
    for k, piece in enumerate(pieces):
        if r["kind"] != "table" and (len(piece) < MIN_CHARS or len(piece.split()) < 8):
            continue
        header = f"{r['doc_short']}" + (f" — {r['section']}" if r["section"] else "") + f" (page {r['page']})"
        chunks.append(dict(
            chunk_id=f"{r['doc_id']}-p{r['page']:03d}-{'t' if r['kind']=='table' else 'c'}{len(chunks):04d}",
            doc_id=r["doc_id"], doc_short=r["doc_short"], page=int(r["page"]), section=r["section"],
            kind=r["kind"], method=r["method"], text=piece, header=header,
            source=source_label(r["doc_short"], r["section"], r["page"]), n_chars=len(piece)))

chunks_df = pd.DataFrame(chunks)
chunks_df.to_json(PROC_DIR / "chunks.jsonl", orient="records", lines=True, force_ascii=False)

stats = chunks_df.groupby("doc_short")["n_chars"].describe()[["count", "min", "50%", "75%", "max"]].round(0).astype(int)
stats.columns = ["chunks", "min_chars", "median_chars", "p75_chars", "max_chars"]
display(stats)
per_unit = chunks_df.groupby(["doc_id", "page", "section"]).size()
print("total chunks:", len(chunks_df), "| pages/sections split into several chunks:", int((per_unit > 1).sum()),
      "| pages kept whole:", int((per_unit == 1).sum()), "| largest chunk:", int(chunks_df["n_chars"].max()), "chars")
print("\nExample chunk:\n", chunks_df[chunks_df.section == "Temple of Edfu"].iloc[0]["text"])

,chunks,min_chars,median_chars,p75_chars,max_chars
doc_short,,,,,
Alexandria Guide,73,63,908,1107,1784
Aswan Guide,19,101,863,1007,1461
Egypt Map & Guide,93,210,841,1089,1277
Egypt Tourism Guide,49,62,680,1063,1622


total chunks: 234 | pages/sections split into several chunks: 8 | pages kept whole: 153 | largest chunk: 1784 chars

Example chunk:
 Temple of Edfu

By visiting the temple of Edfu, which lies on the west bank of the Nile, you can explore one of the most completely preserved ancient Egyptian temples. Built during the Ptolemaic Period (305-30 BC), it is dedicated to the gods Horus, his wife Hathor, and their son Harsomtus (Horus Sema-Tawy 'Horus-Who-Unites-The-Two-Lands'). One of the reasons for the exceptional preservation of this temple is the fact that most of it was buried under sand, until Auguste Mariette cleared it in 1860. and restoration of some parts Fascinating scenes adorn the walls, including those in the inner and outer hypostyle halls that show the temple foundation rituals being performed by the king. And make sure you don't miss the scenes depicting the Beautiful Feast of Behdet, for which the cult statue of the goddess hathor sailed along distance from her temple of Hor

### Why these sizes?

* **1,200 characters ≈ 250–300 tokens.** Measured on this corpus: the median chunk is ≈ 870 characters, 75 % of chunks are under ≈ 1,090, and only **8 of 161 pages/sections needed splitting** — most chunks *are* whole pages. That size is small enough that one chunk answers one question ("ticket price of Edfu") without dragging in unrelated text, and large enough to keep a paragraph plus its facts together.
* **Budget check.** The retriever returns 5 chunks → at most ~6,000 characters ≈ 1,500 tokens of context, which fits comfortably in the 4,096-token window used for the local Llama 3.2 model together with the prompt and answer.
* **200-character overlap (~17 %)** is about two sentences. It is enough to keep a fact that straddles a boundary in at least one chunk, without inflating the index (overlap only exists on the few pages that need splitting).
* **Why keep pages up to 1,800?** The Aswan pages (title + description + address/hours/prices ≈ 1,000–1,600 chars) must stay in one piece, otherwise the price line would lose the name of the site it belongs to. A fixed 1,200-character window would cut exactly there.
* **Why not pure fixed-size windows?** They ignore the document structure: a window would routinely mix the end of one destination with the start of the next.

# 2.3 Embeddings & Vector Store

* **Embedding model:** `nomic-embed-text` served by Ollama (768 dimensions, runs locally, needs the `search_document:` / `search_query:` prefixes it was trained with). Using Ollama for both the notebook *and* the backend means the backend needs no PyTorch.
* **Vector store:** Chroma, persisted to disk with cosine distance. The backend loads this folder directly.

In [13]:
# client = ollama.Client(host=OLLAMA_HOST)


# def _model_names():
#     resp = client.list()
#     models = getattr(resp, "models", None) or resp["models"]
#     return [getattr(m, "model", None) or m["model"] for m in models]


# try:
#     available = _model_names()
# except Exception as e:
#     raise RuntimeError(f"Cannot reach Ollama at {OLLAMA_HOST}. Start it (`ollama serve`) and retry. Original error: {e}")
# for needed in (EMBED_MODEL, LLM_MODEL):
#     if not any(n.split(":")[0] == needed.split(":")[0] for n in available):
#         raise RuntimeError(f"Model '{needed}' is not installed. Run:  ollama pull {needed}")
# print("Ollama OK. Installed models:", available)


# def embed(texts, prefix, batch=4):
#     out = []
#     for i in range(0, len(texts), batch):
#         resp = client.embed(model=EMBED_MODEL, input=[prefix + t for t in texts[i:i + batch]])
#         out += list(getattr(resp, "embeddings", None) or resp["embeddings"])
#     return out


# t0 = time.time()
# docs_to_embed = [f"{h}\n{t}" for h, t in zip(chunks_df["header"], chunks_df["text"])]
# embeddings = embed(docs_to_embed, DOC_PREFIX)
# print(f"embedded {len(embeddings)} chunks in {time.time() - t0:.0f}s, dimension = {len(embeddings[0])}")
import time
import requests
import ollama

# 1. إنشاء العميل للتحقق من النماذج
client = ollama.Client(host=OLLAMA_HOST)

def _model_names():
    resp = client.list()
    models = getattr(resp, "models", None) or resp["models"]
    return [getattr(m, "model", None) or m["model"] for m in models]

try:
    available = _model_names()
except Exception as e:
    raise RuntimeError(f"Cannot reach Ollama at {OLLAMA_HOST}. Start it (`ollama serve`) and retry. Original error: {e}")

for needed in (EMBED_MODEL, LLM_MODEL):
    if not any(n.split(":")[0] == needed.split(":")[0] for n in available):
        raise RuntimeError(f"Model '{needed}' is not installed. Run:  ollama pull {needed}")

print("Ollama OK. Installed models:", available)

# 2. دالة التضمين الآمنة باستخدام requests (لتجنب انهيار الذاكرة نهائياً)
def embed(texts, prefix, batch=1):
    out = []
    url = f"{OLLAMA_HOST}/api/embed"
    
    for i, t in enumerate(texts):
        full_text = prefix + t
        if len(full_text) > 2000:
            full_text = full_text[:2000]  # اقتطاع النصوص الطويلة حمايةً للذاكرة
            
        payload = {
            "model": EMBED_MODEL,
            "input": [full_text]
        }
        
        response = requests.post(url, json=payload)
        if response.status_code != 200:
            raise RuntimeError(f"Error from Ollama API: {response.text}")
            
        res_json = response.json()
        embeddings_batch = res_json.get("embeddings")
        out.append(embeddings_batch[0])
        time.sleep(0.05)  # راحة قصيرة لتفريغ الضغط
        
    return out

# 3. تشغيل التضمين وحساب الوقت
t0 = time.time()
docs_to_embed = [f"{h}\n{t}" for h, t in zip(chunks_df["header"], chunks_df["text"])]
embeddings = embed(docs_to_embed, DOC_PREFIX)
print(f"embedded {len(embeddings)} chunks in {time.time() - t0:.0f}s, dimension = {len(embeddings[0])}")

Ollama OK. Installed models: ['nomic-embed-text:latest', 'llama3.2:latest']
embedded 234 chunks in 715s, dimension = 768


In [14]:
if STORE_DIR.exists():
    shutil.rmtree(STORE_DIR)                       # always rebuild from scratch: no stale chunks
chroma = chromadb.PersistentClient(path=str(STORE_DIR), settings=Settings(anonymized_telemetry=False))
collection = chroma.create_collection(COLLECTION, metadata={"hnsw:space": "cosine"}, embedding_function=None)
collection.add(
    ids=chunks_df["chunk_id"].tolist(),
    embeddings=embeddings,
    documents=chunks_df["text"].tolist(),
    metadatas=chunks_df[["doc_id", "doc_short", "page", "section", "kind", "method", "source"]].to_dict("records"),
)
print("vector store persisted at:", STORE_DIR, "| chunks stored:", collection.count())

vector store persisted at: C:\Users\user\Downloads\rag-assistant-app\data\vector_store | chunks stored: 234


# 2.4 Retrieval & Prompting

* `retrieve()` embeds the question (with the query prefix) and returns the top-*k* chunks with their cosine distance.
* The **prompt template** numbers each passage `[1] … [5]`, labels it with its source, and instructs the model to answer *only* from those passages and to cite them.
* **Citation-style grounding:** the answer must contain `[n]` markers; the code maps them back to source labels (`Aswan Guide – Temple of Edfu (p. 4)`) and renumbers them so `[1]`, `[2]` in the answer match the numbered source list shown to the user.
* **Refusal guard:** if the best chunk is farther than a calibrated distance threshold (calibrated in 2.6), the model is not even called and the assistant says it could not find the answer.

In [15]:
NOT_FOUND = "I couldn't find that in the provided documents."

SYSTEM_PROMPT = (
    "You answer questions about travel in Egypt using ONLY the numbered context passages you are given.\n"
    "Rules:\n"
    "1. Use only facts stated in the passages. Never add outside knowledge.\n"
    "2. After every fact, cite the passage number in square brackets, for example [1] or [2][3].\n"
    "3. Copy prices, times, distances and numbers exactly as written in the passages.\n"
    "4. If passages give different values, mention both and cite each one.\n"
    f"5. If the passages do not contain the answer, reply exactly: {NOT_FOUND}\n"
    "Keep the answer short (1-4 sentences)."
)


def retrieve(question, k=TOP_K):
    q = embed([question], QUERY_PREFIX)[0]
    res = collection.query(query_embeddings=[q], n_results=k, include=["documents", "metadatas", "distances"])
    return [dict(text=d, source=m["source"], doc_id=m["doc_id"], page=m["page"], distance=float(dist))
            for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]


def build_messages(question, hits):
    context = "\n\n".join(f"[{i}] ({h['source']})\n{h['text']}" for i, h in enumerate(hits, 1))
    user = f"Context passages:\n{context}\n\nQuestion: {question}\nAnswer:"
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]


def cite(answer, hits):
    """Map [n] citations to source labels and renumber them so [1], [2]... match the returned source list."""
    if NOT_FOUND.lower()[:20] in answer.lower():
        return answer, []
    used = []
    for n in (int(x) for x in re.findall(r"\[(\d+)\]", answer)):
        if 1 <= n <= len(hits) and n not in used:
            used.append(n)
    if not used:                                                  # model forgot to cite: fall back to the top passage
        return re.sub(r"\s*\[\d+\]", "", answer).strip(), [hits[0]["source"]]
    sources, number = [], {}
    for n in used:
        label = hits[n - 1]["source"]
        if label not in sources:
            sources.append(label)
        number[n] = sources.index(label) + 1
    out = re.sub(r"\[(\d+)\]", lambda m: f"[{number[int(m.group(1))]}]" if int(m.group(1)) in number else "", answer)
    return re.sub(r"\s{2,}", " ", out).strip(), sources


def ask(question, k=TOP_K, max_distance=None):
    """Full pipeline: retrieve -> prompt -> LLM -> answer with cited sources."""
    hits = retrieve(question, k)
    if not hits or (max_distance is not None and hits[0]["distance"] > max_distance):
        return dict(answer=NOT_FOUND, sources=[], hits=hits, gated=True)
    resp = client.chat(model=LLM_MODEL, messages=build_messages(question, hits),
                       options={"temperature": 0.1, "num_ctx": 4096})
    answer = (getattr(getattr(resp, "message", None), "content", None) or resp["message"]["content"]).strip()
    answer, cited = cite(answer, hits)
    return dict(answer=answer, sources=cited, hits=hits, gated=False)

### Test questions
Sixteen questions with known answers drawn from all four documents: facts inside OCR'd text (prices, hours), a parsed table (distance, temperature), plain prose, a question whose answer appears in two guides, and **three out-of-scope questions** that a grounded assistant must refuse.
Each in-scope question lists the text that must appear together **inside one retrieved chunk** (`ctx`) and in the answer (`ans`); a group passes if *any* of its alternatives matches, and *all* groups must pass.

In [16]:
TESTS = [
    # question, ctx groups (retrieval must contain), ans groups (answer must contain), category
    ("How much is a visitor ticket to the Temple of Edfu?",                   [["Visitor 450 EGP"]],                 [["450"]],                                  "OCR: ticket price"),
    ("What are the opening hours of the Nubian Museum?",                      [["9:00 am – 5:00 pm"]],               [["9:00", "9 am", "9am"]],                   "OCR: opening hours"),
    ("How much does the Sun Alignment ticket at Abu Simbel cost?",            [["Sun Alignment: Visitor 900 EGP"]],  [["900"]],                                  "OCR: 2nd price table"),
    ("Which gods is the temple of Kom Ombo dedicated to?",                    [["Sobek"]],                           [["Sobek"], ["Horus", "Harwer"]],            "prose, two guides"),
    ("How much would the Unfinished Obelisk have weighed and who commissioned it?", [["1168 tons"]],                 [["1168", "1,168"], ["Hatshepsut"]],         "OCR: prose facts"),
    ("What are the emergency phone numbers for the police and an ambulance in Egypt?", [["122"], ["123"]],           [["122"], ["123"]],                          "list of numbers"),
    ("What is Egypt's currency and how is it divided?",                       [["100 piasters"]],                    [["pound"], ["100 piaster", "100 piastre"]], "prose"),
    ("What is the distance between Siwa Oasis and Alexandria in kilometres?", [["Siwa Oasis to Alexandria: 590 km", "Alexandria to Siwa Oasis: 590 km"]], [["590"]],                         "table → facts"),
    ("What is the average maximum temperature in Aswan in August?",           [["August: min 25, max 41"]],          [["41"]],                                   "table → facts"),
    ("What are the opening hours of Qaitbey Citadel?",                        [["Qaitbey"], ["9am-4:30pm"]],          [["4:30", "4.30"]],                          "Alexandria: info block"),
    ("Who founded Alexandria and in which year?",                             [["Alexander"], ["332"]],              [["Alexander"], ["332"]],                     "prose, two guides"),
    ("Which oasis is known as the 'Northern Oasis'?",                         [["Northern Oasis"]],                  [["Bahariyya", "Bahariya"]],                 "semantic, Map poster"),
    ("What phone number can I call for complaints about the Aswan sites?",    [["19654"]],                           [["19654"]],                                "OCR: last page"),
    ("What is the ticket price for the Great Pyramid of Giza?",               None,                                  None,                                       "out of scope"),
    ("Who won the 2022 FIFA World Cup?",                                      None,                                  None,                                       "out of scope"),
    ("What is the best restaurant in Paris?",                                 None,                                  None,                                       "out of scope"),
]


def norm(s):
    return re.sub(r"\s+", " ", s.lower())


def groups_ok(groups, text):
    t = norm(text)
    return all(any(norm(alt) in t for alt in g) for g in groups)


# ---- retrieval-only test (no LLM): is the needed text inside the top-k retrieved chunks?
rows = []
for q, ctx, ans, cat in TESTS:
    hits = retrieve(q)
    rank = None
    if ctx:
        for i, h in enumerate(hits, 1):
            if groups_ok(ctx, h["text"]):
                rank = i; break
        all_ctx = rank is not None
    rows.append(dict(question=q, category=cat, top1_source=hits[0]["source"], top1_distance=round(hits[0]["distance"], 3),
                     answer_chunk_rank=rank if ctx else "n/a",
                     needed_text_in_top_k=("yes" if all_ctx else "NO") if ctx else "n/a"))
retrieval_test = pd.DataFrame(rows)
display(retrieval_test)
in_scope = retrieval_test[retrieval_test.needed_text_in_top_k != "n/a"]
print(f"needed text found in top-{TOP_K} for {(in_scope.needed_text_in_top_k == 'yes').sum()}/{len(in_scope)} in-scope questions")

,question,category,top1_source,top1_distance,answer_chunk_rank,needed_text_in_top_k
0,How much is a visitor ticket to the Temple of Edfu?,OCR: ticket price,Aswan Guide – Temple of Edfu (p. 4),0.286,1,yes
1,What are the opening hours of the Nubian Museum?,OCR: opening hours,Aswan Guide – Nubian Museum (p. 10),0.214,1,yes
2,How much does the Sun Alignment ticket at Abu Simbel cost?,OCR: 2nd price table,Aswan Guide – Abu Simbel (p. 18),0.375,1,yes
3,Which gods is the temple of Kom Ombo dedicated to?,"prose, two guides",Egypt Tourism Guide (p. 34),0.181,1,yes
4,How much would the Unfinished Obelisk have weighed and who commissioned it?,OCR: prose facts,Aswan Guide – Unfinished Obelisk (p. 12),0.238,1,yes
5,What are the emergency phone numbers for the police and an ambulance in Egypt?,list of numbers,Alexandria Guide (p. 96),0.165,1,yes
6,What is Egypt's currency and how is it divided?,prose,Alexandria Guide (p. 94),0.194,1,yes
7,What is the distance between Siwa Oasis and Alexandria in kilometres?,table → facts,Egypt Map & Guide – Distances from Siwa Oasis (p. 2),0.148,1,yes
8,What is the average maximum temperature in Aswan in August?,table → facts,Egypt Map & Guide – Temperatures in Aswan (p. 2),0.186,1,yes
9,What are the opening hours of Qaitbey Citadel?,Alexandria: info block,Alexandria Guide (p. 29),0.330,1,yes


needed text found in top-5 for 13/13 in-scope questions


# 2.5 Vision Component

*Not applicable — this project follows the **Core Track** (text-only RAG). No YOLO / computer-vision model is used.*
(The only image-related work is OCR of the scanned Aswan guide, described in 2.1.)

# 2.6 Evaluation

The full pipeline (retrieve → LLM → cited answer) is run on all 16 test questions.

**Refusal threshold calibration.** In-scope questions should retrieve something close (small cosine distance); out-of-scope questions should not. The threshold is set between the two groups:
midway between the worst in-scope top-1 distance and the best out-of-scope top-1 distance (if they overlap, just above the worst in-scope distance). It is stored in the exported config and used by the backend.

**Scoring (automatic, deterministic):**
* *retrieval relevant* — one of the retrieved chunks contains the needed text;
* *cited* — the answer contains a valid `[n]` citation;
* *answer correct* — the expected facts appear in the answer;
* *correct* = all three; for out-of-scope questions, correct = the assistant refused instead of inventing an answer.

In [17]:
ins  = retrieval_test[retrieval_test.category != "out of scope"]["top1_distance"]
outs = retrieval_test[retrieval_test.category == "out of scope"]["top1_distance"]
worst_in, best_out = float(ins.max()), float(outs.min())
if best_out > worst_in:
    MAX_DISTANCE = round((worst_in + best_out) / 2, 3)
    print(f"classes separable: in-scope max = {worst_in:.3f}, out-of-scope min = {best_out:.3f}")
else:
    MAX_DISTANCE = round(worst_in + 0.02, 3)
    print(f"classes overlap (in-scope max {worst_in:.3f} >= out-of-scope min {best_out:.3f}): threshold just above in-scope max; "
          "the prompt's refusal rule has to catch the rest")
print("MAX_DISTANCE =", MAX_DISTANCE)

results = []
for q, ctx, ans, cat in TESTS:
    t0 = time.time()
    out = ask(q, max_distance=MAX_DISTANCE)
    dt_s = time.time() - t0
    if ctx:
        retrieval_ok = any(groups_ok(ctx, h["text"]) for h in out["hits"])
        cited = bool(out["sources"]) and not out["gated"]
        answer_ok = groups_ok(ans, out["answer"])
        correct = retrieval_ok and cited and answer_ok
    else:
        retrieval_ok = cited = answer_ok = None
        correct = NOT_FOUND[:20].lower() in out["answer"].lower()
    results.append(dict(question=q, category=cat, retrieved_source=out["hits"][0]["source"],
                        cited_sources="; ".join(out["sources"]) or "—", answer=out["answer"].replace("\n", " "),
                        retrieval_relevant=retrieval_ok, cited=cited, answer_matches=answer_ok,
                        correct=bool(correct), seconds=round(dt_s, 1)))
    print(("PASS" if correct else "FAIL"), "|", q)
eval_df = pd.DataFrame(results)

classes overlap (in-scope max 0.375 >= out-of-scope min 0.353): threshold just above in-scope max; the prompt's refusal rule has to catch the rest
MAX_DISTANCE = 0.395
PASS | How much is a visitor ticket to the Temple of Edfu?
PASS | What are the opening hours of the Nubian Museum?
PASS | How much does the Sun Alignment ticket at Abu Simbel cost?
PASS | Which gods is the temple of Kom Ombo dedicated to?
PASS | How much would the Unfinished Obelisk have weighed and who commissioned it?
PASS | What are the emergency phone numbers for the police and an ambulance in Egypt?
PASS | What is Egypt's currency and how is it divided?
PASS | What is the distance between Siwa Oasis and Alexandria in kilometres?
FAIL | What is the average maximum temperature in Aswan in August?
PASS | What are the opening hours of Qaitbey Citadel?
FAIL | Who founded Alexandria and in which year?
FAIL | Which oasis is known as the 'Northern Oasis'?
PASS | What phone number can I call for complaints about the Aswan si

In [18]:
def mark(v):
    return "✅" if v else "❌"

table = pd.DataFrame({
    "#": range(1, len(eval_df) + 1),
    "Question": eval_df["question"],
    "Retrieved source (top-1)": eval_df["retrieved_source"],
    "Answer": eval_df["answer"].str.slice(0, 220),
    "Correct?": eval_df["correct"].map(mark),
})
display(table)

n_in  = sum(1 for t in TESTS if t[1])
n_out = len(TESTS) - n_in
acc_in  = int(eval_df[eval_df.category != "out of scope"]["correct"].sum())
acc_out = int(eval_df[eval_df.category == "out of scope"]["correct"].sum())
ret_ok  = int(eval_df["retrieval_relevant"].fillna(False).sum())
cited_n = int(eval_df["cited"].fillna(False).sum())
summary_md = (f"**In-scope questions answered correctly:** {acc_in}/{n_in}  \n"
              f"**Out-of-scope questions correctly refused:** {acc_out}/{n_out}  \n"
              f"**Retrieval relevant (needed text in top-{TOP_K}):** {ret_ok}/{n_in}  \n"
              f"**Answers with a valid citation:** {cited_n}/{n_in}  \n"
              f"**Models:** embeddings `{EMBED_MODEL}`, LLM `{LLM_MODEL}` · chunk size {CHUNK_SIZE}/{CHUNK_OVERLAP} · refusal threshold {MAX_DISTANCE}")
print(summary_md.replace("  \n", "\n"))

# ---- write docs/evaluation_results.md and refresh the results block of README.md
md_table = table.to_markdown(index=False)
stamp = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
(DOCS_DIR / "evaluation_results.md").write_text(
    f"# Evaluation results\n\n_Generated by `notebooks/rag_pipeline.ipynb` on {stamp}._\n\n{summary_md}\n\n{md_table}\n", encoding="utf-8")
eval_df.to_csv(DOCS_DIR / "evaluation_results.csv", index=False)

readme = ROOT / "README.md"
if readme.exists():
    txt = readme.read_text(encoding="utf-8")
    block = f"<!-- EVAL_TABLE_START -->\n_Generated {stamp} by the notebook._\n\n{summary_md}\n\n{md_table}\n<!-- EVAL_TABLE_END -->"
    new = re.sub(r"<!-- EVAL_TABLE_START -->.*?<!-- EVAL_TABLE_END -->", lambda m: block, txt, flags=re.S)
    readme.write_text(new, encoding="utf-8")
    print("README.md evaluation section updated.")

,#,Question,Retrieved source (top-1),Answer,Correct?
0,1,How much is a visitor ticket to the Temple of Edfu?,Aswan Guide – Temple of Edfu (p. 4),The visitor ticket to the Temple of Edfu costs 450 EGP [1].,✅
1,2,What are the opening hours of the Nubian Museum?,Aswan Guide – Nubian Museum (p. 10),"The opening hours of the Nubian Museum are 9:00 am – 5:00 pm, with extended hours on Thursdays and Fridays...",✅
2,3,How much does the Sun Alignment ticket at Abu Simbel cost?,Aswan Guide – Abu Simbel (p. 18),The Sun Alignment ticket at Abu Simbel costs 900 EGP for visitors and 450 EGP for students. [1][2],✅
3,4,Which gods is the temple of Kom Ombo dedicated to?,Egypt Tourism Guide (p. 34),"The temple of Kom Ombo is dedicated to two deities: the crocodile god Sobek, and the falcon god Harwer (Ho...",✅
4,5,How much would the Unfinished Obelisk have weighed and who commissioned it?,Aswan Guide – Unfinished Obelisk (p. 12),The Unfinished Obelisk would have weighed 1168 tons and was commissioned by Hatshepsut [1][2].,✅
5,6,What are the emergency phone numbers for the police and an ambulance in Egypt?,Alexandria Guide (p. 96),The emergency phone numbers for the police and an ambulance in Egypt are: * Police: 122 [1][2] * Ambulance...,✅
6,7,What is Egypt's currency and how is it divided?,Alexandria Guide (p. 94),"Egypt's currency is the Egyptian pound, which is divided into 100 piasters [1][2][3].",✅
7,8,What is the distance between Siwa Oasis and Alexandria in kilometres?,Egypt Map & Guide – Distances from Siwa Oasis (p. 2),The distance between Siwa Oasis and Alexandria is 590 km [1][2].,✅
8,9,What is the average maximum temperature in Aswan in August?,Egypt Map & Guide – Temperatures in Aswan (p. 2),The average maximum temperature in Aswan in August is 40 °C [1].,❌
9,10,What are the opening hours of Qaitbey Citadel?,Alexandria Guide (p. 29),The opening hours of Qaitbey Citadel are 9am-4:30pm [1].,✅


**In-scope questions answered correctly:** 10/13
**Out-of-scope questions correctly refused:** 2/3
**Retrieval relevant (needed text in top-5):** 13/13
**Answers with a valid citation:** 13/13
**Models:** embeddings `nomic-embed-text`, LLM `llama3.2` · chunk size 1200/200 · refusal threshold 0.395
README.md evaluation section updated.


C:\Users\user\AppData\Local\Temp\ipykernel_19476\3059340515.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ret_ok  = int(eval_df["retrieval_relevant"].fillna(False).sum())
C:\Users\user\AppData\Local\Temp\ipykernel_19476\3059340515.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cited_n = int(eval_df["cited"].fillna(False).sum())


In [19]:
# Failing questions (if any) with the details needed for the failure analysis below
failed = eval_df[~eval_df["correct"]]
if failed.empty:
    print("All test questions passed.")
else:
    display(failed[["question", "retrieved_source", "retrieval_relevant", "cited", "answer_matches", "answer"]])

,question,retrieved_source,retrieval_relevant,cited,answer_matches,answer
8,What is the average maximum temperature in Aswan in August?,Egypt Map & Guide – Temperatures in Aswan (p. 2),True,True,False,The average maximum temperature in Aswan in August is 40 °C [1].
10,Who founded Alexandria and in which year?,Egypt Tourism Guide (p. 39),True,True,False,Alexandria was founded by Alexander the Great in 331 B.C.E [1].
11,Which oasis is known as the 'Northern Oasis'?,Egypt Map & Guide (p. 1),True,True,False,"The Northern Oasis is approximately 4000 km2 desert depression, set in black volcanic rock, and is reachab..."
13,What is the ticket price for the Great Pyramid of Giza?,Egypt Tourism Guide (p. 16),None,None,None,The ticket price for the Great Pyramid of Giza is not mentioned in the provided passages.


### Failure cases and mitigations

**Problems found while building the corpus — and what fixed them** (these were the real risks to answer quality):

| Failure case | Symptom | Mitigation |
|---|---|---|
| Aswan guide is a scan (no text layer) | 0 characters extracted | OCR (RapidOCR at 300 dpi) |
| OCR reads the ticket table left-to-right | Which number is *Visitor* and which *Student* is lost | Prices assigned to the *Visitor*/*Student* header above them using box coordinates → `Ticket prices: Visitor 450 EGP, Student 230 EGP.` |
| OCR glues words / confuses letters | `Donotmiss`, `Nckhbct`, `sccn` | Dictionary-guided repair (letter-confusion substitution + word segmentation) + explicit manual-fix list |
| OCR detector skips a whole line | Sentence silently missing | Gap detection + re-OCR of the band |
| Map poster tables are loose numbers | Temperatures / distances not retrievable or misread | Parsed into one sentence-chunk per city, validated (row lengths, symmetry) |
| Curved map labels extract as single letters | Hundreds of junk one-character chunks | Drop boxes narrower than 6 pt |
| Page cut splits a site from its price | Price chunk without the site name | Keep pages ≤ 1,800 chars whole; embed each chunk with a header containing the section title |
| Out-of-scope questions | LLM answers from its own knowledge | Distance-threshold gate + strict "answer only from passages / refuse" prompt |
| Two guides disagree (e.g. Cairo–Aswan is 950 km in one guide, 982 km in the distance table; obelisk height 41 m vs 42 m) | Confident but one-sided answer | Prompt rule 4: report both values and cite each |

**Observed on this run** — see the table above and the failing-question output. *(Edit this paragraph after running the notebook: name the questions that failed, whether the cause was retrieval (needed text not in the top-k), generation (fact retrieved but mis-copied / not cited) or scoring (correct answer phrased differently), and what you changed.)*

# 2.7 Export

Copy the persisted vector store into `backend/data/vector_store/` together with a `config.json` that records everything the backend needs to reproduce the query side of the pipeline: embedding model and prefixes, collection name, chunk parameters, top-k and the refusal threshold. **The backend never rebuilds anything at request time.**

In [20]:
if BACKEND_STORE.exists():
    shutil.rmtree(BACKEND_STORE)
shutil.copytree(STORE_DIR, BACKEND_STORE)

config = dict(
    collection_name=COLLECTION,
    embedding_model=EMBED_MODEL, doc_prefix=DOC_PREFIX, query_prefix=QUERY_PREFIX,
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, page_keep_limit=PAGE_KEEP_LIMIT, min_chars=MIN_CHARS,
    top_k=TOP_K, max_distance=MAX_DISTANCE, distance_metric="cosine",
    n_chunks=int(collection.count()), llm_model_used_in_eval=LLM_MODEL,
    sources=sorted(chunks_df["doc_short"].unique().tolist()),
    created_at=dt.datetime.now().isoformat(timespec="seconds"),
)
(BACKEND_STORE / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(config, indent=2, ensure_ascii=False))

# ---- verify: open the exported copy with a brand-new client, exactly as the backend does
check = chromadb.PersistentClient(path=str(BACKEND_STORE), settings=Settings(anonymized_telemetry=False))
col2 = check.get_collection(config["collection_name"])
assert col2.count() == config["n_chunks"]
q = embed(["How much is a student ticket to Philae?"], config["query_prefix"])[0]
res = col2.query(query_embeddings=[q], n_results=1, include=["metadatas", "distances"])
print("\nexport verified — exported store answers a query:", res["metadatas"][0][0]["source"], f"(distance {res['distances'][0][0]:.3f})")
print("size on disk:", round(sum(f.stat().st_size for f in BACKEND_STORE.rglob('*') if f.is_file()) / 1e6, 1), "MB")

{
  "collection_name": "egypt_travel_guides",
  "embedding_model": "nomic-embed-text",
  "doc_prefix": "search_document: ",
  "query_prefix": "search_query: ",
  "chunk_size": 1200,
  "chunk_overlap": 200,
  "page_keep_limit": 1800,
  "min_chars": 60,
  "top_k": 5,
  "max_distance": 0.395,
  "distance_metric": "cosine",
  "n_chunks": 234,
  "llm_model_used_in_eval": "llama3.2",
  "sources": [
    "Alexandria Guide",
    "Aswan Guide",
    "Egypt Map & Guide",
    "Egypt Tourism Guide"
  ],
  "created_at": "2026-09-22T22:41:24"
}

export verified — exported store answers a query: Aswan Guide – Temple of Philae (p. 14) (distance 0.418)
size on disk: 3.6 MB
